In [ ]:
!pip install duckdb
!pip install great_expectations

source

import pandas as pd
import duckdb
import great_expectations as gx

  Using cached great_expectations-1.17.2-py3-none-any.whl.metadata (9.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 12.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [great_expectations]reat_expectations]


### 1. EXTRACT: Load the messy data

In [14]:
from pathlib import Path

input_path = Path.cwd() / "messy_ecommerce_data.csv"
print(f"Reading from: {input_path.resolve()}")
raw_data = pd.read_csv(input_path)
print(f"Loaded {len(raw_data)} rows")

Reading from: /content/messy_ecommerce_data.csv
Loaded 100 rows


In [22]:
print(raw_data)

    transaction_id  customer_age  purchase_amount     status
0                1          17.0        56.066057  completed
1                2          25.0        20.801862    pending
2                3          25.0        27.870870    UNKNOWN
3                4          85.0       104.529818  completed
4                5          34.0        84.526490  completed
..             ...           ...              ...        ...
95              96         120.0        46.001390    pending
96              97          85.0        -3.533023  completed
97              98          34.0        97.638868     failed
98              99          25.0        -5.190403     failed
99             100          17.0        57.172096  completed

[100 rows x 4 columns]


---

### 2. TRANSFORM: Clean using DuckDB SQL

In [17]:
clean_query = """
    SELECT 
        transaction_id,
        CAST(customer_age AS INTEGER) as customer_age,
        ROUND(purchase_amount, 2) as purchase_amount,
        status
    FROM raw_data
    WHERE customer_age IS NOT NULL 
    AND purchase_amount > 0
    AND status IN ('completed', 'pending', 'failed')
"""

In [21]:
clean_data = duckdb.query(clean_query).df()
print(clean_data)

    transaction_id  customer_age  purchase_amount     status
0                1            17            56.07  completed
1                2            25            20.80    pending
2                4            85           104.53  completed
3                5            34            84.53  completed
4                7            25            48.62     failed
..             ...           ...              ...        ...
56              94            85            52.73  completed
57              95            25            22.07    pending
58              96           120            46.00    pending
59              98            34            97.64     failed
60             100            17            57.17  completed

[61 rows x 4 columns]


In [23]:
print(f"Rows before cleaning: {len(raw_data)} | Rows after cleaning: {len(clean_data)}")

Rows before cleaning: 100 | Rows after cleaning: 61


### 3. VALIDATE

In [57]:
from pathlib import Path

print("Validating data quality...")
validated_data = clean_data.loc[clean_data["customer_age"].between(18, 99)].copy()
validated_data = validated_data.reset_index(drop=True)
print(f"Rows before validation filter: {len(clean_data)} | Rows after validation filter: {len(validated_data)}")
context = gx.get_context()
pandas_source = context.data_sources.add_pandas(name="my_pandas")
batch = pandas_source.read_dataframe(validated_data)
validator = context.get_validator(
    batch=batch,
    create_expectation_suite_with_name="clean_data_suite",
)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpzwwxbv44' for ephemeral docs site


Validating data quality...
Rows before validation filter: 61 | Rows after validation filter: 33


In [58]:
# Strict data quality rules
validator.expect_column_values_to_not_be_null("transaction_id")

  warnings.warn(



Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "batch_id": "my_pandas-#ephemeral_pandas_asset",
      "column": "transaction_id"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 33,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [59]:
validator.expect_column_values_to_be_between("customer_age", min_value=18, max_value=99)

  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "my_pandas-#ephemeral_pandas_asset",
      "column": "customer_age",
      "min_value": 18.0,
      "max_value": 99.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 33,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [60]:
validator.expect_column_values_to_be_in_set("status", ["completed", "pending", "failed"])

  warnings.warn(



Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "my_pandas-#ephemeral_pandas_asset",
      "column": "status",
      "value_set": [
        "completed",
        "pending",
        "failed"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 33,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [61]:
validation_result = validator.validate()

Calculating Metrics:   0%|          | 0/19 [00:00<?, ?it/s]

In [62]:
if validation_result.success:
    print("Validation passed. Saving gold_ecommerce_data.csv...")
    gold_path = Path.cwd() / "gold_ecommerce_data.csv"
    validated_data.to_csv(gold_path, index=False)
    print(f"Saved to {gold_path.resolve()}")
else:
    print("Validation failed. Pipeline stopped.")
    print(validation_result)

print(f"Validation success: {validation_result.success}")

Validation passed. Saving gold_ecommerce_data.csv...
Saved to /content/gold_ecommerce_data.csv
Validation success: True
